In [ ]:
import pandas as pd
import gcsfs
from google.cloud import bigquery
from google.colab import auth
auth.authenticate_user()

# --- Configuration ---
bucket_base_path = "gs://dcceew-input-data/Palaeo Data"
meta_folder_path = f"{bucket_base_path}/Metadata/"
Run_FOA56 = True
Run_Rain = True
Run_Mwet = True

# Set your version tag here (e.g., "_v2", "_test_run", or "" for no tag)
version_tag = "_v2"

# Initialize client
project_id = "paleo-interpolation"
dataset_id = "Paleo_Data"
client = bigquery.Client(project=project_id)

variables = []

if run_FOA56:
    variables.append("FOA56")
if run_Rain:
    variables.append("Rain")
if run_Mwet:
    variables.append("Mwet")

# ==========================================
# PART 1: Upload Metadata to BigQuery
# ==========================================
print("Uploading Metadata to BigQuery...")
fs = gcsfs.GCSFileSystem()
csv_files = fs.glob(f"{meta_folder_path}*.csv")
df_list = []

for file in csv_files:
    # ADDED: low_memory=False stops the DtypeWarning for messy CSVs
    df = pd.read_csv(f"gs://{file}", low_memory=False)

    if df.empty:
        continue

    rename_dict = {col: 'Latitude (Degrees South)' for col in df.columns if 'Latitude' in col}
    rename_dict.update({col: 'Longitude (Degrees East)' for col in df.columns if 'Longitude' in col})
    df = df.rename(columns=rename_dict)

    cols_to_keep = ['Station ID', 'RWS Region']
    try:
        # Added .copy() to prevent Pandas SettingWithCopy warnings
        df = df[cols_to_keep].copy()

        # FIX: Drop any rows where Station ID is missing BEFORE converting to integer
        df = df.dropna(subset=['Station ID'])

        # Now it is safe to format
        df['Station ID'] = df['Station ID'].astype(float).astype(int).astype(str)
        df['Station_Region_ID'] = df['RWS Region'].astype(str) + '_' + df['Station ID']
        df_list.append(df)
    except KeyError:
        continue

md_df = pd.concat(df_list, ignore_index=True).drop_duplicates(subset=['Station_Region_ID'])

# Upload metadata to a permanent table for the JOIN later
meta_table_id = f"{project_id}.{dataset_id}.Master_Metadata{version_tag}"
client.load_table_from_dataframe(
    md_df[['Station_Region_ID']],
    meta_table_id,
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
).result()
print(f"Metadata loaded: {len(md_df)} unique stations.")

# ==========================================
# PART 2: The BigQuery Pushdown
# ==========================================

# Extract the unique region names from the metadata we processed in Part 1
unique_regions = md_df['RWS Region'].unique()

for var in variables:
    print(f"\n================ Processing Variable: {var} ================")

    ext_table_id = f"{project_id}.{dataset_id}.{var}_external_staging"
    main_table_id = f"{project_id}.{dataset_id}.{var}_Data{version_tag}"

    # 1. Mount GCS files as an External Table
    print("Mounting GCS files as an External Table...")

    # Generate a list of URIs dynamically. Each URI now only has ONE asterisk.
    source_uris_list = [f"{bucket_base_path}/{region}/{var}/*_SILO_{var}.csv" for region in unique_regions]

    external_config = bigquery.ExternalConfig("CSV")
    external_config.source_uris = source_uris_list
    external_config.options.skip_leading_rows = 1

    external_config.schema = [
        bigquery.SchemaField("Date_Str", "STRING"),
        bigquery.SchemaField(var, "FLOAT64")
    ]
    external_config.options.ignore_unknown_values = True

    table = bigquery.Table(ext_table_id)
    table.external_data_configuration = external_config

    client.delete_table(ext_table_id, not_found_ok=True)
    client.create_table(table)

    # 2. Execute massive distributed ELT via SQL
    print(f"Running distributed transformation and load into {main_table_id}...")

    sql = f"""
    CREATE OR REPLACE TABLE `{main_table_id}`
    PARTITION BY RANGE_BUCKET(Decade, GENERATE_ARRAY(0, 10000, 10))
    CLUSTER BY Station_Region_ID, Date
    AS
    WITH parsed_data AS (
        SELECT
            Date_Str AS Date,
            {var},
            CAST(SPLIT(Date_Str, '/')[SAFE_OFFSET(2)] AS INT64) AS Year,
            CAST(SPLIT(Date_Str, '/')[SAFE_OFFSET(1)] AS INT64) AS Month,
            CAST(SPLIT(Date_Str, '/')[SAFE_OFFSET(0)] AS INT64) AS Day,
            REGEXP_EXTRACT(_FILE_NAME, r'/Palaeo Data/([^/]+)/') AS Region,
            CAST(CAST(REGEXP_EXTRACT(_FILE_NAME, r'/([0-9]+)_SILO') AS INT64) AS STRING) AS S_id
        FROM `{ext_table_id}`
        WHERE {var} IS NOT NULL
    ),
    formatted_data AS (
        SELECT
            Date,
            CONCAT(Region, '_', S_id) AS Station_Region_ID,
            Year,
            Month,
            Day,
            CAST(FLOOR(Year / 10) * 10 AS INT64) AS Decade,
            {var}
        FROM parsed_data
    )
    SELECT f.*
    FROM formatted_data f
    INNER JOIN `{meta_table_id}` m
    ON f.Station_Region_ID = m.Station_Region_ID
    """

    query_job = client.query(sql)
    query_job.result()

    client.delete_table(ext_table_id, not_found_ok=True)

    print(f"Finished processing {var} successfully!")

print("\nPipeline complete! All data transformed and loaded into BigQuery.")

Uploading Metadata to BigQuery...
Metadata loaded: 1500 unique stations.

================ Processing Variable: FAO56 ================
Mounting GCS files as an External Table...
Running distributed transformation and load into paleo-interpolation.Paleo_Data.FAO56_Data_v2...
Finished processing FAO56 successfully!

================ Processing Variable: Mwet ================
Mounting GCS files as an External Table...
Running distributed transformation and load into paleo-interpolation.Paleo_Data.Mwet_Data_v2...
Finished processing Mwet successfully!

================ Processing Variable: Rain ================
Mounting GCS files as an External Table...
Running distributed transformation and load into paleo-interpolation.Paleo_Data.Rain_Data_v2...
Finished processing Rain successfully!

Pipeline complete! All data transformed and loaded into BigQuery.
